# B2-024 — Practice p17

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: optional-colab-l4` · seed `20261022` · the CPU correctness path below is required and is what CI runs; the `Accelerator extension` section is optional, runs only on a Colab L4, and is never executed by CI  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** semi-supervised-pseudo-labeling

## Task

Certify that noisy-student self-training beats the fixed labelled-only baseline on `shapes_ssl`, selecting only on labelled validation data.

**Data.** `xl, yl = capstone_data.train_rows("shapes_ssl")` (30 labelled images), `xv, yv = capstone_data.val_rows("shapes_ssl")` (30 labelled validation images), `pool = capstone_data.unlabelled_rows()` (600 images, labels never available). The images are `(N, 1, 8, 8)` float32, three shape classes as in Session 4.

**Student.** `class NoisyStudent(nn.Module)` with constructor `NoisyStudent(noise_std)` storing `self.noise_std` and then creating `self.cnn = capstone_data.ShapeCNN()`; `forward(x)` adds `self.noise_std * torch.randn_like(x)` to `x` **only when** `self.training` and `noise_std > 0`, then returns `self.cnn(x)`.
**Config.** Every device, precision, size, and schedule knob is read from one config dict with the keys of the supplied `CPU_CONFIG`, which is the path CI runs; the accelerator extension changes only the dictionary.
Write `train_student(x, y, noise_std, config)`: `device = torch.device(config["device"])`; `torch.manual_seed(SEED)`; `model = NoisyStudent(noise_std).to(device)`; register the seam hook (below); `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`; `model.train()`; move `x, y` to `device`; exactly `config["steps_per_round"]` (**150** on the CPU) updates of `F.cross_entropy(model(xb), yb)`, where `(xb, yb)` is the full batch `(x, y)` when `config["batch_size"]` is `None` (the CPU path) and otherwise `(x[idx], y[idx])` for `idx = torch.randint(0, len(x), (config["batch_size"],))` drawn from the global generator. The forward pass and the loss run inside `torch.autocast(device_type=device.type, dtype=config["amp_dtype"])` when `config["amp_dtype"]` is not `None`, with no autocast otherwise. (Each update runs, in this order: `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()`; no scheduler, no gradient clipping, no accumulation, no early stopping.) Return `model.eval()`.
With `noise_std = 0.0` on `(xl, yl)` and `CPU_CONFIG` this is exactly the committed baseline recipe.

**Seam.** The hook is a forward pre-hook on the `NoisyStudent` that, on the **first** training-mode call of each `train_student`, records `capstone_data.split_of("shapes_ssl", row)` for every row it receives (the clean rows, before the noise). Every recorded value must be `"train"` or `"unlabelled"`.

**Self-training.** Implement `select_pseudo_labels` with the p09 contract. The pseudo-labelling pool is `pool` when `config["extra_pool"] == 0` (the CPU path), and otherwise `pool` concatenated with the images of `capstone_data.simulate("shapes_ssl", config["extra_pool"], seed=2)`, **discarding their labels**. For each threshold `tau` in `config["thresholds"]` (`(0.7, 0.9)` on the CPU): start with `teacher = train_student(xl, yl, 0.0, config)`; for each round cap in `config["round_caps"]` (`(100, 200, 200)` on the CPU): compute `probs = F.softmax(teacher(pool), dim=1)` under `torch.no_grad()` with the teacher in eval mode, select `idx, labels = select_pseudo_labels(probs, tau, cap)`, and set `teacher = train_student(torch.cat([xl, pool[idx]]), torch.cat([yl, labels]), config["student_noise"], config)` (`student_noise` is 0.3 on the CPU). Record each round's pseudo-label class counts and the final (third-round) student's validation accuracy.

**Selection.** Choose `tau` by the final student's validation accuracy; ties go to the larger `tau`. Also report two validation ablations: the baseline (`noise_std = 0.0`, labelled only) and noise only (`train_student(xl, yl, config["student_noise"], config)`).

**Evaluation protocol.** Fit only on rows from `capstone_data.train_rows("shapes_ssl")` (and any other rows this statement names), make every choice on `capstone_data.val_rows("shapes_ssl")`, and reach the locked test rows only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="shapes_ssl")`, called exactly once, after all fitting and selection. Assert `capstone_data.test_call_count("shapes_ssl") == 0` immediately before that call and `== 1` after it.
`predict_fn` returns the selected student's `argmax` labels (int64) in eval mode under `torch.no_grad()`.

**Certify.** Test accuracy `>= capstone_data.baseline_score("shapes_ssl").score + 0.03` (the committed baseline scores `0.8100` at the frozen seed, so the bar is `0.8400`); every pseudo-label count per class is at most that round's cap; the seam values are all in `{"train", "unlabelled"}`; the test-call protocol.
Then, in at most five sentences, interpret the validation ablations and the class counts in terms of confirmation bias.
The `0.03` margin is fixed by the plan; at the frozen seed the reference run clears it with several points of slack.

**Required answer-check assertions.** Every certificate item, `len(idx)` never exceeding `3 * cap`, and the chosen `tau` being one of the two candidates.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

MARGIN = 0.03
CPU_CONFIG = {"device": "cpu", "amp_dtype": None, "thresholds": (0.7, 0.9), "round_caps": (100, 200, 200),
              "steps_per_round": 150, "batch_size": None, "student_noise": 0.3, "extra_pool": 0}

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Accelerator extension

*Optional; Colab L4 only; never executed by CI and never required for credit.*

On a Colab L4, the same loop scales up by passing `L4_CONFIG` in place of `CPU_CONFIG` to the same `train_student` / self-training code. It has exactly the keys of `CPU_CONFIG`, so only the dictionary changes: `batch_size = 1024` switches on mini-batch sampling with the global generator, and `extra_pool = 50000` enlarges the pool with images simulated from the same process, whose labels are discarded.

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "thresholds": (0.7, 0.8, 0.9), "round_caps": (1000, 5000, 10000, 20000),
             "steps_per_round": 2000, "batch_size": 1024, "student_noise": 0.3, "extra_pool": 50000}
```

**What to observe.** Validation accuracy per round as the pool grows (does it keep rising, or does confirmation bias set in?); per-class pseudo-label counts; and the noise-only ablation at the larger scale.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.